# Unified Model Comparison

This notebook is the central forecasting comparison for the project. All models forecast the same target, `international_arrivals`, using the same 2012-2022 training period and 2023-2025 test period.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import CLEAN_SEGMENTS, FIGURES, TABLES, MAIN_TARGET, SEGMENT_COLUMNS, SERIES_COLORS, SERIES_LABELS
from src.plotting import annotate_events, save_figure, set_academic_style
set_academic_style()
df = pd.read_csv(CLEAN_SEGMENTS, parse_dates=["date"]).set_index("date").sort_index()
df.index.freq = "MS"

from src.models.evaluation import evaluate_target, metrics
from src.config import TRAIN_END

## Train/Test Split

In [ ]:
train = df.loc[:TRAIN_END, MAIN_TARGET]
test = df.loc[pd.Timestamp(TRAIN_END) + pd.offsets.MonthBegin(1):, MAIN_TARGET]
train.index.min(), train.index.max(), test.index.min(), test.index.max(), len(train), len(test)

Observation -> the test period is entirely post-reopening. Statistical implication -> model comparison evaluates recovery-period generalization. Tourism implication -> the benchmark is directly relevant to current demand planning.

## Fit and Compare Models

In [ ]:
rows, results, forecasts = evaluate_target(df[MAIN_TARGET], MAIN_TARGET)
comparison = pd.DataFrame(rows)
ok = comparison[comparison["status"].eq("ok")].copy()
best = ok.sort_values(["sMAPE", "RMSE"]).head(1)
TABLES.mkdir(parents=True, exist_ok=True)
comparison.to_csv(TABLES / "all_model_comparison.csv", index=False)
best.to_csv(TABLES / "best_model_summary.csv", index=False)
comparison.round(2)

Observation -> the benchmark ranks models on identical data and metrics. Statistical implication -> differences reflect model structure rather than different targets or samples. Tourism implication -> model selection can be tied to a single operational forecasting objective.

## Forecast Comparison

In [ ]:
fig, ax = plt.subplots()
ax.plot(df.loc["2018":].index, df.loc["2018":, MAIN_TARGET], color=SERIES_COLORS[MAIN_TARGET], label="Observed")
for model in forecasts["model"].unique():
    sub = forecasts[forecasts["model"].eq(model)].copy()
    sub["date"] = pd.to_datetime(sub["date"])
    ax.plot(sub["date"], sub["forecast"], label=model, alpha=0.9)
ax.set_title("Forecast Comparison for International Tourist Arrivals")
ax.set_ylabel("Monthly arrivals")
ax.legend(fontsize=8)
save_figure(fig, FIGURES / "model_comparison_forecast.png")
plt.show()

Observation -> forecast paths diverge during the recovery period. Statistical implication -> post-shock dynamics are sensitive to model assumptions. Tourism implication -> forecast choice affects capacity and marketing decisions.

## Metric Comparison

In [ ]:
metric_cols = ["MAE", "RMSE", "MAPE", "sMAPE"]
plot_df = ok.set_index("model")[metric_cols]
fig, axes = plt.subplots(2, 2, figsize=(9, 6))
for ax, metric in zip(axes.ravel(), metric_cols):
    vals = plot_df[metric].sort_values()
    ax.bar(vals.index, vals.values, color="#7f7f7f")
    ax.set_title(metric)
    ax.tick_params(axis="x", rotation=35)
fig.suptitle("Forecast Accuracy Metrics Across Main Models", y=1.02)
save_figure(fig, FIGURES / "model_comparison_metrics.png")
plt.show()

Observation -> the ranking can differ across absolute and percentage metrics. Statistical implication -> no single metric fully summarizes forecast quality. Tourism implication -> forecast evaluation should match the planning loss function.

## XGBoost Feature Importance

In [ ]:
if "XGBoost" in results and "feature_importance" in results["XGBoost"]:
    imp = results["XGBoost"]["feature_importance"].head(15).sort_values()
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(imp.index, imp.values, color="#4e79a7")
    ax.set_title("XGBoost Feature Importance")
    ax.set_xlabel("Importance")
    save_figure(fig, FIGURES / "xgboost_feature_importance.png")
    plt.show()
else:
    print("XGBoost feature importance unavailable because the model did not run successfully.")

Observation -> XGBoost importance identifies which lag, rolling, calendar, and event variables drive the machine-learning benchmark. Statistical implication -> predictive structure is restricted to anti-leakage features. Tourism implication -> interpretable lag dependence can support near-term monitoring without using same-month segment information.